# TD – Construction de la collection entreprise

Nous disposons de l'export KBO Open Data, composé de huit fichiers CSV montés sur le volume Docker, ainsi que d'une base MongoDB vide. L'objectif de ce TD est de construire progressivement un pipeline permettant d'importer ces fichiers dans MongoDB, puis de les assembler afin de produire une collection entreprise. Chaque document représentera une entreprise et contiendra ses établissements ainsi que ses succursales sous forme de structures imbriquées.

# 0) Objectif du TD
**Le modèle de données KBO**

La KBO (Kruispuntbank van Ondernemingen), registre officiel des entreprises belges, diffuse ses données sous forme de plusieurs fichiers CSV indépendants, à la manière d'un export de base de données relationnelle.

Les données sont organisées autour de trois types d'entités :

Niveau	Fichier	Clé principale	Description
Entreprise	enterprise.csv	EnterpriseNumber	Personne juridique
Établissement	establishment.csv	EstablishmentNumber	Unité opérationnelle belge (magasin, siège, usine, etc.)
Succursale	branch.csv	Id	Implantation belge d'une entreprise étrangère

Quatre autres fichiers apportent des informations complémentaires sur ces entités :

denomination.csv
address.csv
contact.csv
activity.csv

Tous utilisent une même clé de rattachement : EntityNumber. Selon les cas, cette colonne contient le numéro d'une entreprise, d'un établissement ou d'une succursale.

Cette caractéristique est au cœur du TD. Les fichiers de détails ne distinguent pas le type d'entité auquel ils appartiennent : ils ne contiennent qu'un identifiant. La même logique de jointure pourra donc être appliquée aux trois niveaux, ce qui permettra de développer une fonction réutilisable.

**Du modèle relationnel au modèle documentaire**

Dans le modèle actuel, récupérer toutes les informations d'une entreprise nécessite de consulter plusieurs fichiers distincts, soit jusqu'à huit lectures différentes.

L'objectif de la couche Bronze est de réaliser ce travail une seule fois, lors d'un traitement batch, afin de produire un document MongoDB autonome regroupant toutes les informations d'une entreprise.

---

**La couche Bronze**

La couche Bronze conserve fidèlement les données de la source sans les modifier.

Aucune transformation métier n'est réalisée :

les codes restent inchangés (Status = "AC", TypeOfAddress = "REGO"),
les colonnes multilingues sont conservées (MunicipalityNL, MunicipalityFR),
les valeurs vides restent inchangées.

Le rôle de cette couche est uniquement de regrouper les informations afin de construire des documents complets. Cette approche garantit une couche facilement rejouable et fidèle aux données d'origine. Les traitements d'interprétation et de nettoyage seront réalisés dans la couche Silver.

**Configuration**

Le notebook utilise trois paramètres de configuration, tous lus depuis les variables d'environnement avec une valeur par défaut. Cette approche permet d'exécuter le même notebook sans modification aussi bien dans le conteneur Docker du TD (/data/kbo, mongodb://mongo:27017) que sur une machine locale.

In [ ]:
import csv
import os
import time
from itertools import islice
from pathlib import Path

import pymongo

def resolve_data_dir() -> Path:
    """Premier repertoire candidat contenant reellement l'export KBO."""
    candidates = (os.getenv("KBO_DATA_DIR"), "/data/kbo", Path.cwd(), Path.cwd().parent)
    for candidate in candidates:
        if candidate and Path(candidate).joinpath("meta.csv").exists():
            return Path(candidate)
    raise FileNotFoundError(
        "Export KBO introuvable. Definissez KBO_DATA_DIR sur le dossier "
        f"contenant meta.csv (candidats testes : {candidates})."
    )

DATA_DIR = resolve_data_dir()
MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")
DB_NAME = os.getenv("MONGO_DB", "kbo")

client = pymongo.MongoClient(MONGO_URI, socketTimeoutMS=None)
db = client[DB_NAME]

print("donnees :", DATA_DIR)
print("mongodb :", MONGO_URI, "->", DB_NAME)
print("serveur :", client.server_info()["version"])

Un coup d'oeil a la volumetrie avant de commencer : c'est elle qui dicte toutes
les decisions techniques qui suivent.

In [ ]:
CSV_FILES = {
    # fichier            collection cible      champ promu en _id
    "meta.csv":          ("kbo_meta",          None),
    "code.csv":          ("kbo_code",          None),
    "enterprise.csv":    ("kbo_enterprise",    "EnterpriseNumber"),
    "establishment.csv": ("kbo_establishment", None),
    "branch.csv":        ("kbo_branch",        None),
    "denomination.csv":  ("kbo_denomination",  None),
    "address.csv":       ("kbo_address",       None),
    "contact.csv":       ("kbo_contact",       None),
    "activity.csv":      ("kbo_activity",      None),
}

print(f"{'fichier':<20}{'taille':>10}   colonnes")
for filename in CSV_FILES:
    path = DATA_DIR / filename
    with path.open(encoding="utf-8", newline="") as handle:
        header = next(csv.reader(handle))
    print(f"{filename:<20}{path.stat().st_size / 1e6:>9,.0f}M   {', '.join(header)}")

# 1) Import des fichiers CSV dans MongoDB

Les fichiers de l'export KBO sont volumineux. L'objectif de cette étape est de développer une fonction capable d'importer un fichier CSV dans une collection MongoDB par lots (batch), puis de l'utiliser pour charger les huit fichiers de l'export, à raison d'une collection par fichier.

**Principe du chargement**

Certains fichiers, comme activity.csv, contiennent plusieurs dizaines de millions de lignes. Il est donc impossible de les charger entièrement en mémoire.

Pour réaliser un import efficace, on s'appuie sur trois principes :

csv.DictReader lit le fichier progressivement, ligne par ligne.
iter_batches regroupe les lignes en lots afin de limiter l'utilisation de la mémoire.
insert_many(ordered=False) insère chaque lot en une seule opération, ce qui améliore les performances.

**Bonnes pratiques**

Quelques paramètres permettent de garantir un import fiable :

utiliser encoding="utf-8" afin de conserver correctement les caractères accentués ;
utiliser newline="" pour laisser le module csv gérer correctement les retours à la ligne ;
définir EnterpriseNumber comme clé _id pour enterprise.csv, ce qui évite les doublons et rend l'import réexécutable.

**Reprise du chargement**

La fonction doit également pouvoir reprendre un traitement interrompu :

si la collection est déjà complète, elle est laissée inchangée ;
si elle est partiellement remplie, elle est vidée puis rechargée.

Cette approche permet de relancer le pipeline sans devoir recommencer l'ensemble de l'import.

In [ ]:
BATCH_SIZE = 50_000

def iter_batches(iterable, size: int):
    """Decoupe un iterable en listes de `size` elements, sans le materialiser."""
    iterator = iter(iterable)
    while batch := list(islice(iterator, size)):
        yield batch


def count_csv_rows(csv_path: Path) -> int:
    """Nombre de lignes de donnees (en-tete exclu), lu en streaming."""
    with csv_path.open("r", encoding="utf-8", newline="") as handle:
        return sum(1 for _ in handle) - 1


def load_csv_to_collection(csv_path: Path, collection_name: str, *,
                           id_field: str | None = None,
                           batch_size: int = BATCH_SIZE,
                           skip_if_complete: bool = True) -> int:
    """Charge un CSV dans une collection MongoDB, par batch, a memoire constante."""
    collection = db[collection_name]
    expected = count_csv_rows(csv_path)

    if skip_if_complete and collection.estimated_document_count() == expected:
        print(f"-- {collection_name:<20} deja complet ({expected:>12,}) - ignore")
        return expected

    collection.drop()                      # rechargement = snapshot complet
    inserted, started = 0, time.perf_counter()

    with csv_path.open("r", encoding="utf-8", newline="") as handle:
        for batch in iter_batches(csv.DictReader(handle), batch_size):
            if id_field:                   # cle naturelle -> _id
                for document in batch:
                    document["_id"] = document[id_field]
            collection.insert_many(batch, ordered=False)
            inserted += len(batch)

    elapsed = time.perf_counter() - started
    print(f"OK {collection_name:<20} {inserted:>12,} docs en {elapsed:7.1f}s "
          f"({inserted / elapsed:>9,.0f}/s)")
    return inserted

On applique la fonction aux 9 fichiers de l'export (les 8 fichiers de donnees
plus `meta.csv`, qui documente la date du snapshot).

In [ ]:
started = time.perf_counter()
total = sum(
    load_csv_to_collection(DATA_DIR / filename, collection_name, id_field=id_field)
    for filename, (collection_name, id_field) in CSV_FILES.items()
)
print(f"\nTOTAL {total:,} documents en {(time.perf_counter() - started) / 60:.1f} min")

### Verification

On compare le nombre de documents charges au nombre de lignes des CSV : c'est le
controle d'integrite minimal apres une ingestion.

In [ ]:
print(f"{'collection':<20}{'documents':>14}{'lignes CSV':>14}   etat")
for filename, (collection_name, _) in CSV_FILES.items():
    loaded = db[collection_name].count_documents({})
    rows = count_csv_rows(DATA_DIR / filename)
    print(f"{collection_name:<20}{loaded:>14,}{rows:>14,}   "
          f"{'OK' if loaded == rows else 'ECART'}")

print("\nsnapshot :", {d["Variable"]: d["Value"] for d in db.kbo_meta.find()})

# 2) Création des index pour les jointures

Afin d'optimiser les opérations de jointure ($lookup), il est nécessaire de créer des index sur les champs utilisés comme clés de liaison entre les collections. Sans ces index, MongoDB devrait parcourir l'ensemble des documents à chaque jointure, ce qui ralentirait fortement le pipeline.

**Champs à indexer**

Les collections de détails (kbo_denomination, kbo_address, kbo_contact et kbo_activity) doivent être indexées sur le champ EntityNumber, utilisé pour rattacher les informations aux entreprises, établissements et succursales.

Les collections kbo_establishment et kbo_branch doivent quant à elles être indexées sur EnterpriseNumber, afin de relier chaque établissement ou succursale à son entreprise.

**Collections sans index supplémentaire **

La collection kbo_enterprise ne nécessite pas d'index additionnel, car elle constitue la collection principale du pipeline et son identifiant (_id, correspondant à EnterpriseNumber) est déjà indexé par MongoDB.

La collection kbo_code n'est pas utilisée dans cette étape ; elle servira lors de la construction de la couche Silver.

Quand créer les index ?

Les index sont créés après le chargement des données. Cette approche est plus performante, car maintenir un index pendant chaque insertion est beaucoup plus coûteux que de le construire une seule fois sur une collection déjà remplie.

In [ ]:
JOIN_INDEXES = {
    "kbo_denomination":  "EntityNumber",
    "kbo_address":       "EntityNumber",
    "kbo_contact":       "EntityNumber",
    "kbo_activity":      "EntityNumber",
    "kbo_establishment": "EnterpriseNumber",
    "kbo_branch":        "EnterpriseNumber",
}

for collection_name, field in JOIN_INDEXES.items():
    started = time.perf_counter()
    index_name = db[collection_name].create_index(field)
    print(f"{collection_name:<20} {index_name:<22} {time.perf_counter() - started:>7.1f}s")

In [ ]:
stats = db.command("dbStats", scale=1024 * 1024)
print(f"donnees : {stats['dataSize']:>8,.0f} Mo")
print(f"stockage: {stats['storageSize']:>8,.0f} Mo  (compression zstd)")
print(f"index   : {stats['indexSize']:>8,.0f} Mo")

---

## 3) Rejoindre les details d'une entite

Trois niveaux d'entites : entreprise, etablissement, branche -- ont chacun
besoin des memes quatre informations complementaires : leurs denominations,
leurs adresses, leurs contacts et leurs activites. Ces quatre informations
vivent chacune dans leur propre collection, et s'y rattachent toujours de la
meme maniere, quel que soit le type d'entite concerne.

Ecrivez une fonction reutilisable qui, etant donne le nom du champ a utiliser
comme cle de jointure du cote de l'entite courante, construit les etapes
d'agregation necessaires pour rattacher ces quatre informations. Cette fonction
sera appelee trois fois dans la suite du TD, une fois par niveau d'entite, avec
a chaque fois un nom de champ different.

### L'asymetrie qui rend la fonction reutilisable

Le rattachement d'un detail a son entite est **asymetrique** :

- du **cote du detail**, le champ est *toujours* `EntityNumber` — les quatre
  collections ont ete concues comme ca ;
- du **cote de l'entite**, le champ **change de nom** a chaque niveau :
  `EnterpriseNumber`, `EstablishmentNumber`, ou `Id`.

Toute la variabilite tient donc dans **un seul parametre**, `primary_key`. C'est
exactement ce que demande l'enonce : une fonction, trois appels.

La fonction renvoie une **liste de 4 etapes** (et non un seul `$lookup`), ce qui
permet de la deballer avec `*` a l'endroit voulu du pipeline.

| Appel | `primary_key` | Contexte |
|---|---|---|
| `_detail_lookups("EnterpriseNumber")` | `EnterpriseNumber` | question 6, au niveau entreprise |
| `_detail_lookups("EstablishmentNumber")` | `EstablishmentNumber` | question 5, dans le sous-pipeline etablissement |
| `_detail_lookups("Id")` | `Id` | question 4, dans le sous-pipeline succursale |

> **Pourquoi les succursales recoivent-elles aussi contacts et activites ?**
> Une succursale n'en a jamais dans les faits. Mais appliquer les 4 jointures
> partout garde la fonction uniforme et produit simplement `contacts: []` et
> `activities: []`. Un tableau vide coute quelques octets ; une exception dans
> le code coute bien plus cher a maintenir. C'est la couche silver qui
> supprimera ces cles pour de bon.

In [ ]:
DETAIL_SOURCES = (
    ("kbo_denomination", "denominations"),
    ("kbo_address",      "addresses"),
    ("kbo_contact",      "contacts"),
    ("kbo_activity",     "activities"),
)

def _detail_lookups(primary_key: str) -> list[dict]:
    """Les 4 etapes rattachant a une entite ses informations complementaires.

    `primary_key` est le champ qui identifie l'entite courante et dont la valeur
    se retrouve dans la colonne `EntityNumber` des 4 collections de details :
    `EnterpriseNumber`, `EstablishmentNumber` ou `Id` selon le niveau.
    """
    return [
        {"$lookup": {"from": source,
                     "localField": primary_key,
                     "foreignField": "EntityNumber",
                     "as": alias}}
        for source, alias in DETAIL_SOURCES
    ]


for stage in _detail_lookups("EnterpriseNumber"):
    print(stage)

---

## 4) Rejoindre les succursales

Une succursale represente la presence en Belgique d'une entreprise etrangere.
Chaque succursale est rattachee a une entreprise, et a, comme vu a la question
3, ses propres denominations et adresses (mais jamais de contacts ni
d'activites).

Ecrivez une fonction qui construit l'etape d'agregation permettant de rattacher,
pour chaque entreprise, la liste de ses succursales completes, chaque
succursale devant elle-meme deja porter ses propres denominations et adresses,
obtenues via la fonction de la question 3. Le lien entre une entreprise et ses
succursales ne se fait pas sur le meme champ que celui utilise a l'interieur
d'une succursale pour aller chercher ses propres denominations et adresses :
il faudra donc correler explicitement les deux niveaux.

### Deux cles differentes dans une seule etape

C'est le point delicat signale par l'enonce. Pour la succursale `9.000.006.626`
de l'entreprise `0257.883.408`, **deux** cles interviennent :

```
entreprise 0257.883.408
   │
   │  (a) lien entreprise -> succursale : kbo_branch.EnterpriseNumber == "0257.883.408"
   ▼
succursale  Id = "9.000.006.626"
   │
   │  (b) lien succursale -> ses details : kbo_address.EntityNumber == "9.000.006.626"
   ▼
adresse de la succursale
```

La forme courte du `$lookup` (`localField` / `foreignField`) ne sait exprimer
qu'**une** correlation. Il faut donc la forme longue :

- **`let`** capture la valeur du document parent — ici `EnterpriseNumber` — et
  la publie dans une variable `$$enterprise_number` ;
- **`pipeline`** s'execute *dans le contexte de la collection jointe*
  (`kbo_branch`), ou l'on retrouve cette variable ;
- **`$match` + `$expr`** realise la correlation (a) : c'est `$expr` qui permet
  de comparer un champ a une variable, ce qu'un `$match` ordinaire ne sait pas faire ;
- une fois dans ce sous-pipeline, on est « chez la succursale » : les etapes de
  la question 3 appliquees a `Id` realisent la correlation (b).

> **Le piege** : ecrire `_detail_lookups("EnterpriseNumber")` dans le
> sous-pipeline. On rattacherait alors a chaque succursale les denominations et
> adresses de **son entreprise mere**, pas les siennes. Le pipeline
> fonctionnerait sans la moindre erreur et produirait des donnees fausses — le
> genre de bug qu'on ne voit qu'en relisant une ligne de resultat.

La fonction est ecrite une fois de facon generique : la question 5 (les
etablissements) a exactement la meme structure, seuls la collection source et le
nom de la cle enfant changent.

In [ ]:
def _children_lookup(*, source: str, child_key: str, alias: str) -> dict:
    """Rattache a une entreprise ses entites filles, deja enrichies de leurs details.

    - `source`    : collection des filles (`kbo_branch` ou `kbo_establishment`)
    - `child_key` : cle propre de la fille (`Id` ou `EstablishmentNumber`),
                    passee a `_detail_lookups` a l'interieur du sous-pipeline
    - `alias`     : nom du tableau produit dans le document entreprise
    """
    return {"$lookup": {
        "from": source,
        # (a) correlation explicite entreprise -> fille
        "let": {"enterprise_number": "$EnterpriseNumber"},
        "pipeline": [
            {"$match": {"$expr": {"$eq": ["$EnterpriseNumber", "$$enterprise_number"]}}},
            # (b) chaque fille va chercher ses propres details, avec SA cle
            *_detail_lookups(child_key),
        ],
        "as": alias,
    }}


def _branch_lookup() -> dict:
    """Question 4 : les succursales, jointes par `Id`."""
    return _children_lookup(source="kbo_branch", child_key="Id", alias="branches")


import json
print(json.dumps(_branch_lookup(), indent=2))

---

## 5) Rejoindre les etablissements

Meme exercice que la question 4, mais pour les etablissements, les unites
operationnelles d'une entreprise belge.

Comme pour les succursales, chaque etablissement doit deja porter ses propres denominations, adresses, contacts et
activites (obtenus via la fonction de la question 3) avant d'etre rattache a son entreprise.

L'enonce dit « meme exercice », et c'est litteralement vrai : la structure est
identique, seuls deux parametres changent. `_children_lookup` ayant ete ecrite
de facon generique a la question 4, la reponse tient en une ligne.

| | Question 4 | Question 5 |
|---|---|---|
| collection source | `kbo_branch` | `kbo_establishment` |
| cle propre de la fille | `Id` | `EstablishmentNumber` |
| lien vers l'entreprise | `EnterpriseNumber` | `EnterpriseNumber` |
| alias produit | `branches` | `establishments` |

C'est aussi le lookup **le plus couteux du pipeline** : 1,69 million
d'etablissements, chacun declenchant a son tour 4 jointures. A lui seul il
represente l'essentiel du temps d'execution de la question 6.

In [ ]:
def _establishment_lookup() -> dict:
    """Question 5 : les etablissements, joints par `EstablishmentNumber`."""
    return _children_lookup(source="kbo_establishment",
                            child_key="EstablishmentNumber",
                            alias="establishments")


print(json.dumps(_establishment_lookup(), indent=2))

---

## 6) Assembler et executer le pipeline complet

Combinez les fonctions precedentes en un seul pipeline d'agregation, lance sur
la collection contenant les entreprises : les quatre informations
complementaires de l'entreprise elle-meme, puis ses etablissements, puis ses
succursales.

le resultat de ce pipeline doit s'ecrire dans une nouvelle collection,

Executez le pipeline, et afficher le document
complet d'une entreprise qui possede au moins un etablissement, et d'une
entreprise qui possede au moins une succursale.

### Assemblage

Le pipeline se lit comme l'enonce :

```python
[
    *_detail_lookups("EnterpriseNumber"),   # les 4 details de l'entreprise
    _establishment_lookup(),                # puis ses etablissements  (Q5)
    _branch_lookup(),                       # puis ses succursales     (Q4)
    {"$out": "entreprise"},                 # ecriture du resultat
]
```

Deux options d'execution meritent une explication :

- **`$out`** ecrit le resultat dans une nouvelle collection. Il remplace
  atomiquement la collection cible : le pipeline est donc rejouable sans risque
  de doublon, et les lecteurs ne voient jamais d'etat intermediaire.
  (`$merge` serait le choix pour un rafraichissement incremental ; ici on
  reconstruit un snapshot complet, `$out` est plus simple et plus rapide.)
- **`allowDiskUse=True`** autorise les etapes a deborder sur disque au-dela des
  100 Mo de RAM autorises par defaut. Sur ce volume, c'est indispensable.

> **Note de volumetrie** : le document le plus gros correspond a l'entreprise
> `0214.596.464` et ses 1 058 etablissements, soit environ 3 Mo — confortablement
> sous la limite BSON de 16 Mo par document. Cette verification n'est pas
> optionnelle sur un modele imbrique : c'est la contrainte qui decide si une
> denormalisation est viable ou non.

L'execution demande plusieurs dizaines de minutes : 1,95 M d'entreprises et
1,69 M d'etablissements declenchent au total pres de 15 millions de recherches
indexees.

In [ ]:
def build_pipeline(target: str = "entreprise") -> list[dict]:
    """Le pipeline complet de la couche bronze."""
    return [
        *_detail_lookups("EnterpriseNumber"),   # Q3 : details de l'entreprise
        _establishment_lookup(),                # Q5 : etablissements enrichis
        _branch_lookup(),                       # Q4 : succursales enrichies
        {"$out": target},                       # Q6 : ecriture
    ]


pipeline = build_pipeline()
print(f"{len(pipeline)} etapes :",
      [next(iter(stage)) for stage in pipeline])

In [ ]:
started = time.perf_counter()
db.kbo_enterprise.aggregate(pipeline, allowDiskUse=True)
elapsed = time.perf_counter() - started

print(f"collection `entreprise` construite en {elapsed / 60:.1f} min")
print(f"{db.entreprise.count_documents({}):,} documents")

### Une entreprise avec au moins un etablissement

`0403.449.823` : deux etablissements, chacun portant ses propres denominations,
adresses et activites. Noter que `branches` est vide — c'est une entreprise
belge.

In [ ]:
import json

def show(document: dict) -> None:
    print(json.dumps(document, indent=2, ensure_ascii=False, default=str))

show(db.entreprise.find_one({"_id": "0403.449.823"}))

### Une entreprise avec au moins une succursale

`0257.883.408` : une association turque presente en Belgique. Son adresse
d'entreprise est en Turquie, et elle possede une succursale (`branches`) en plus
d'un etablissement.

In [ ]:
show(db.entreprise.find_one({"_id": "0257.883.408"}))

### Controle final

On verifie que le pipeline n'a **perdu personne** (autant de documents que
d'entreprises) et que les entites filles ont bien ete rattachees.

In [ ]:
enterprises = db.kbo_enterprise.count_documents({})
produced = db.entreprise.count_documents({})

print(f"entreprises en entree : {enterprises:>10,}")
print(f"documents produits    : {produced:>10,}   {'OK' if produced == enterprises else 'ECART'}")
print(f"avec >= 1 etablissement: {db.entreprise.count_documents({'establishments.0': {'$exists': True}}):>10,}")
print(f"avec >= 1 succursale   : {db.entreprise.count_documents({'branches.0': {'$exists': True}}):>10,}")
print(f"avec >= 1 activite     : {db.entreprise.count_documents({'activities.0': {'$exists': True}}):>10,}")

stats = db.command("collStats", "entreprise", scale=1024 * 1024)
print(f"\ntaille `entreprise`   : {stats['size']:,.0f} Mo "
      f"({stats['storageSize']:,.0f} Mo sur disque)")
print(f"document moyen        : {stats['avgObjSize'] / 1024:,.1f} Ko")

---

## Bilan

La couche bronze est en place : **un document auto-suffisant par entreprise**,
obtenu en une passe batch, la ou il fallait auparavant 8 lectures dispersees.

Ce qu'il faut retenir des choix techniques :

| Choix | Raison |
|---|---|
| Chargement par batch en streaming | memoire constante sur un fichier de 1,5 Go |
| `_id = EnterpriseNumber` | cle naturelle : index gratuit + chargement idempotent |
| Index avant `$lookup` | sans eux le pipeline ne se termine pas |
| Une fonction `_detail_lookups`, trois appels | les 4 details se rattachent partout de la meme facon |
| `let` + `$expr` pour les filles | deux cles differentes dans une seule etape |
| `$out` | remplacement atomique, pipeline rejouable |

Et ce que le bronze **n'a volontairement pas fait** : traduire `Status="AC"`,
choisir entre `MunicipalityNL` et `MunicipalityFR`, ou dedoublonner les activites
qui reapparaissent sous plusieurs versions NACE. C'est l'objet du second
notebook, la couche **silver**.